# Visibility Check

This notebook validates phase 02 against the loaded tri-state floor plan, writes `02_visibility.npz`, reloads it, and checks the locked LOS behavior on both the real workspace floor plan and small synthetic corner cases.


In [ ]:
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt

from src.common.floorplan import FloorPlanInput, NULL_CELL, OPEN_CELL, SOLID_CELL
from src.planner.main import load_workspace
from src.planner.phase01_candidate_generation import (
    PHASE_ARTIFACT_STEM as PHASE01_ARTIFACT_STEM,
    generate_candidate_generation_artifacts,
    load_candidate_generation_artifacts,
    save_candidate_generation_artifacts,
    validate_candidate_generation_artifacts,
)
from src.planner.phase02_visibility import (
    PHASE_ARTIFACT_STEM,
    PHASE_NAME,
    generate_visibility_artifacts,
    get_diagonal_blocked_target_ordinals,
    get_visible_target_ordinals,
    load_visibility_artifacts,
    save_visibility_artifacts,
    validate_visibility_artifacts,
)


In [ ]:
workspace = load_workspace()

phase01_artifact_path = workspace.artifact_dir / f"{PHASE01_ARTIFACT_STEM}.npz"
if phase01_artifact_path.exists():
    phase01_artifacts = load_candidate_generation_artifacts(phase01_artifact_path)
else:
    phase01_artifacts = generate_candidate_generation_artifacts(workspace.floorplan)
    save_candidate_generation_artifacts(phase01_artifact_path, phase01_artifacts)
validate_candidate_generation_artifacts(workspace.floorplan, phase01_artifacts)

visibility_artifacts = generate_visibility_artifacts(
    workspace.floorplan,
    phase01_artifacts,
)
phase02_artifact_path = workspace.artifact_dir / f"{PHASE_ARTIFACT_STEM}.npz"
save_visibility_artifacts(phase02_artifact_path, visibility_artifacts)
reloaded = load_visibility_artifacts(phase02_artifact_path)
validate_visibility_artifacts(workspace.floorplan, phase01_artifacts, reloaded)

PHASE_NAME, workspace.floorplan.name, phase02_artifact_path


In [ ]:
visible_counts = np.diff(visibility_artifacts.los_candidate_offsets)
diagonal_counts = np.diff(visibility_artifacts.diagonal_candidate_offsets)
summary = {
    "floorplan_name": workspace.floorplan.name,
    "grid_shape": workspace.floorplan.shape,
    "open_cell_count": int(visibility_artifacts.open_cell_count),
    "candidate_count": int(visibility_artifacts.candidate_count),
    "visible_pair_count": int(len(visibility_artifacts.los_target_ordinals)),
    "diagonal_blocked_pair_count": int(len(visibility_artifacts.diagonal_target_ordinals)),
    "mean_visible_targets_per_candidate": float(visible_counts.mean()) if len(visible_counts) else 0.0,
    "max_visible_targets_per_candidate": int(visible_counts.max()) if len(visible_counts) else 0,
}
summary


In [ ]:
def build_floorplan(name: str, rows: list[list[np.int8]]) -> FloorPlanInput:
    grid = np.asarray(rows, dtype=np.int8)
    return FloorPlanInput(
        name=name,
        source_path=Path(f"{name}.png"),
        grid=grid,
        height=int(grid.shape[0]),
        width=int(grid.shape[1]),
        null_cell_count=int(np.count_nonzero(grid == NULL_CELL)),
        open_cell_count=int(np.count_nonzero(grid == OPEN_CELL)),
        solid_cell_count=int(np.count_nonzero(grid == SOLID_CELL)),
    )


def generate_small_visibility(rows: list[list[np.int8]]):
    floorplan = build_floorplan("phase02-small", rows)
    phase01 = generate_candidate_generation_artifacts(floorplan)
    visibility = generate_visibility_artifacts(floorplan, phase01)
    return floorplan, phase01, visibility


# Straight open visibility.
_, _, open_visibility = generate_small_visibility(
    [
        [NULL_CELL, NULL_CELL, NULL_CELL, NULL_CELL],
        [NULL_CELL, OPEN_CELL, OPEN_CELL, NULL_CELL],
        [NULL_CELL, NULL_CELL, NULL_CELL, NULL_CELL],
    ]
)
np.testing.assert_array_equal(get_visible_target_ordinals(open_visibility, 0), np.array([1], dtype=np.int32))

# Solid blockers and null blockers both stop LOS.
for rows in (
    [
        [NULL_CELL, NULL_CELL, NULL_CELL, NULL_CELL, NULL_CELL],
        [NULL_CELL, OPEN_CELL, SOLID_CELL, OPEN_CELL, NULL_CELL],
        [NULL_CELL, NULL_CELL, NULL_CELL, NULL_CELL, NULL_CELL],
    ],
    [
        [NULL_CELL, NULL_CELL, NULL_CELL, NULL_CELL, NULL_CELL],
        [NULL_CELL, OPEN_CELL, NULL_CELL, OPEN_CELL, NULL_CELL],
        [NULL_CELL, NULL_CELL, NULL_CELL, NULL_CELL, NULL_CELL],
    ],
):
    _, _, blocked_visibility = generate_small_visibility(rows)
    np.testing.assert_array_equal(blocked_visibility.los_target_ordinals, np.empty(0, dtype=np.int32))

# Conservative diagonal blocking when both side cells are non-open.
_, _, diagonal_blocked_visibility = generate_small_visibility(
    [
        [NULL_CELL, NULL_CELL, NULL_CELL, NULL_CELL],
        [NULL_CELL, OPEN_CELL, SOLID_CELL, NULL_CELL],
        [NULL_CELL, SOLID_CELL, OPEN_CELL, NULL_CELL],
        [NULL_CELL, NULL_CELL, NULL_CELL, NULL_CELL],
    ]
)
np.testing.assert_array_equal(
    get_diagonal_blocked_target_ordinals(diagonal_blocked_visibility, 0),
    np.array([1], dtype=np.int32),
)

# Diagonal visibility remains valid if one side cell stays open.
_, _, diagonal_open_visibility = generate_small_visibility(
    [
        [NULL_CELL, NULL_CELL, NULL_CELL, NULL_CELL],
        [NULL_CELL, OPEN_CELL, OPEN_CELL, NULL_CELL],
        [NULL_CELL, SOLID_CELL, OPEN_CELL, NULL_CELL],
        [NULL_CELL, NULL_CELL, NULL_CELL, NULL_CELL],
    ]
)
np.testing.assert_array_equal(
    get_visible_target_ordinals(diagonal_open_visibility, 0),
    np.array([1, 2], dtype=np.int32),
)

"small visibility checks passed"


In [ ]:
from matplotlib.patches import Arc


def evenly_sample_target_ordinals(target_ordinals: np.ndarray, max_points: int) -> np.ndarray:
    if len(target_ordinals) <= max_points:
        return target_ordinals
    sample_positions = np.linspace(0, len(target_ordinals) - 1, num=max_points, dtype=np.int64)
    return target_ordinals[np.unique(sample_positions)]


def ray_endpoint(candidate_coord_rc: np.ndarray, angle_deg: float, radius: float) -> tuple[float, float]:
    radians = np.deg2rad(angle_deg)
    end_col = candidate_coord_rc[1] + radius * np.cos(radians)
    end_row = candidate_coord_rc[0] - radius * np.sin(radians)
    return float(end_col), float(end_row)

visible_counts = np.diff(visibility_artifacts.los_candidate_offsets)
sample_candidate_ordinals = np.array([
    np.argsort(visible_counts)[-10],
    np.argsort(visible_counts)[-50],
    np.argsort(visible_counts)[-150],
])

candidate_coords = phase01_artifacts.candidate_cell_coords_rc
open_coords = phase01_artifacts.open_cell_coords_rc
palette = ["#2563eb", "#16a34a", "#9333ea"]

for subplot_index, candidate_ordinal in enumerate(sample_candidate_ordinals.tolist()):
    figure, axis = plt.subplots(figsize=(7, 6))
    workspace.floorplan.plot(
        ax=axis,
        title="",
        show_colorbar=False,
    )
    candidate_coord = candidate_coords[int(candidate_ordinal)]
    visible_target_ordinals = get_visible_target_ordinals(reloaded, int(candidate_ordinal))
    visible_target_coords = open_coords[visible_target_ordinals]
    plotted_target_ordinals = evenly_sample_target_ordinals(visible_target_ordinals, max_points=1000)
    plotted_target_coords = open_coords[plotted_target_ordinals]

    axis.scatter(
        plotted_target_coords[:, 1],
        plotted_target_coords[:, 0],
        s=6,
        c=palette[subplot_index % len(palette)],
        alpha=0.65,
        linewidths=0,
    )
    axis.scatter(
        [candidate_coord[1]],
        [candidate_coord[0]],
        s=12,
        c="#c2410c",
        marker="s",
        linewidths=0,
    )

    axis.set_title(
        f"{workspace.floorplan.name}: candidate {candidate_ordinal}\n"
        f"phase 02 LOS-visible targets {len(visible_target_ordinals)} | sampled {len(plotted_target_ordinals)}\n"
    )
    figure.tight_layout()
    plt.show()
